In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict, deque

ESC = 27
model = YOLO(r'model\yolo11m.pt')#model = YOLO(r'C:\Users\User\Desktop\CW\專題\號誌\Traffic_Sign_Project\single_class_run4')
video_path = r'C:/Users/User/Desktop/CW/專題/main/dataset/sign2.mp4'
clahe_processor = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

# "0": "person",
# "1": "bicycle",
# "2": "car",
# "3": "motorcycle",
# "5": "bus",
# "7": "truck",
VEHICLE_CLASSES = [0, 1, 2, 3, 5, 7]

In [2]:
def adjust_gamma(image, gamma=1.0):
    invGamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** invGamma) * 255
        for i in np.arange(0, 256)]).astype("uint8")

    return cv2.LUT(image, table)

In [3]:
def frame_processing(raw_frame):
    frame = raw_frame.copy()
    
    lab_image = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    
    l_channel, a_channel, b_channel = cv2.split(lab_image)
    
    l_channel_clahe = clahe_processor.apply(l_channel)
    
    merged_lab_image = cv2.merge((l_channel_clahe, a_channel, b_channel))
    
    processed_frame = cv2.cvtColor(merged_lab_image, cv2.COLOR_LAB2BGR)
    
    return processed_frame

In [5]:
def test_vehicle_detector(raw_frame):
    frame = raw_frame.copy()
    (frame_h, frame_w) = frame.shape[:2]

    LEFT_ZONE_BOUNDARY = int(frame_w*0.33)
    RIGHT_ZONE_BOUNDARY = int(frame_w*0.66)

    cv2.line(frame, (LEFT_ZONE_BOUNDARY, 0), (LEFT_ZONE_BOUNDARY, frame_h), (255, 255, 0), 2)
    cv2.line(frame, (RIGHT_ZONE_BOUNDARY, 0), (RIGHT_ZONE_BOUNDARY, frame_h), (255, 255, 0), 2)

    yolo_results = model(frame, classes=vehicle_classes, conf=0.5, verbose=False)

    left_zone_vehicle_count = 0
    mid_zone_vehicle_count = 0
    right_zone_vehicle_count = 0

    for r in yolo_results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            
            center_x = (x1 + x2) // 2
            
            box_color = (0, 255, 0)

            if center_x < LEFT_ZONE_BOUNDARY:
                left_zone_vehicle_count += 1
                box_color = (0, 0, 255)
            elif center_x > RIGHT_ZONE_BOUNDARY:
                right_zone_vehicle_count += 1
                box_color = (0, 0, 255)
            else:
                mid_zone_vehicle_count +=1
                box_color = (0, 165, 255) 

            cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)

    cv2.putText(frame, f'left counter {left_zone_vehicle_count}', (int(frame_w*0.01), frame_h - 60), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (200, 0, 60), 2)
    cv2.putText(frame, f'mid counter {mid_zone_vehicle_count}', (int(frame_w*0.35), frame_h - 60), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (200, 0, 60), 2)
    cv2.putText(frame, f'right counter {right_zone_vehicle_count}', (int(frame_w*0.675), frame_h - 60), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (200, 0, 60), 2)
    
    if left_zone_vehicle_count > 0:
        cv2.putText(frame, "!! VEHICLE LEFT !!", (10, frame_h - 20), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
    else:
        cv2.putText(frame, "LEFT CLEAR", (10, frame_h - 20), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
    if right_zone_vehicle_count > 0:
        cv2.putText(frame, "!! VEHICLE RIGHT !!", (frame_w - 220, frame_h - 20), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
    else:
        cv2.putText(frame, "RIGHT CLEAR", (frame_w - 200, frame_h - 20), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    return frame

In [6]:
# --- 設定參數 ---
HISTORY_LENGTH = 15  # 追蹤歷史長度 (約 0.5 秒)
STABLE_THRESHOLD = 30 # 穩定閾值 (像素)

# 各類別的危險面積閾值
DANGER_AREA_THRESHOLDS = {
    0: 10000,   # person
    1: 10000,   # bicycle
    2: 25000,   # car
    3: 15000,   # motorcycle
    5: 40000,   # bus
    7: 40000    # truck
}

# 歷史紀錄
track_history = defaultdict(lambda: deque(maxlen=HISTORY_LENGTH))
area_history = defaultdict(lambda: deque(maxlen=HISTORY_LENGTH))

def rear_vehicle_tracker(raw_frame):
    frame = raw_frame.copy()
    (frame_h, frame_w) = frame.shape[:2]

    # 1. 執行追蹤
    yolo_results = model.track(frame, classes=VEHICLE_CLASSES, conf=0.5, persist=True, verbose=False)

    if yolo_results[0].boxes.id is None:
        return frame

    # 取出數據 (包含 xyxy, id, 和 cls)
    boxes = yolo_results[0].boxes.xyxy.cpu().numpy()
    track_ids = yolo_results[0].boxes.id.cpu().numpy()
    clss = yolo_results[0].boxes.cls.cpu().numpy() # (新) 取出類別 ID

    # 在迴圈中同時遍歷 box, id, cls
    for box, track_id, cls in zip(boxes, track_ids, clss):
        x1, y1, x2, y2 = map(int, box)
        track_id = int(track_id)
        cls_id = int(cls) # (新) 轉成整數 ID
        
        # --- 計算中心點與面積 ---
        center_x = (x1 + x2) // 2
        center_y = (y1 + y2) // 2
        current_center = (center_x, center_y)
        
        current_area = (x2 - x1) * (y2 - y1)

        # --- 更新歷史紀錄 ---
        track_history[track_id].append(current_center)
        area_history[track_id].append(current_area)

        # 預設狀態
        box_color = (0, 255, 0) # 綠色
        status_text = "Tracking"
        
        # (新) 取得該類別的危險面積閾值
        danger_threshold = DANGER_AREA_THRESHOLDS.get(cls_id, 25000)

        # --- 核心邏輯判斷 ---
        if len(track_history[track_id]) >= 10:
            
            # 1. [穩定點判斷]
            old_center_x, old_center_y = track_history[track_id][-10]
            displacement = np.sqrt((center_x - old_center_x)**2 + (center_y - old_center_y)**2)
            is_stable = displacement < STABLE_THRESHOLD
            
            # 2. [面積趨勢判斷] (正在變大?)
            areas = list(area_history[track_id])
            recent_avg_area = sum(areas[-5:]) / 5
            past_avg_area = sum(areas[-10:-5]) / 5
            is_approaching = recent_avg_area > (past_avg_area * 1.03)
            
            # 3. [危險距離判斷] (新條件: 是否夠近?)
            is_close = current_area > danger_threshold

            # --- 三重條件判定 (AND) ---
            if is_stable and is_approaching and is_close:
                # 條件: 穩定 + 變大 + 夠近 -> 才是真的危險
                status_text = "!! COLLISION !!"
                box_color = (0, 0, 255) # 紅色
                cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 4)
            
            elif is_stable and is_approaching:
                # 穩定 + 變大 + 但還很遠 -> 亮黃燈 (注意)
                status_text = "Approaching..."
                box_color = (0, 255, 255) # 黃色
                cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)

            elif is_stable:
                status_text = "Following"
                box_color = (0, 255, 0)
                cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
            
            else:
                status_text = f"Safe"
                box_color = (0, 255, 0)
                cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)

        else:
            cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)

        # 顯示狀態文字 (包含面積資訊方便除錯)
        label = f"ID:{track_id} {status_text}"
        # 如果想看面積數據可以打開下面這行
        # label += f" ({int(current_area)}/{danger_threshold})"
        
        cv2.putText(frame, label, (x1, y1 - 10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 2)

    return frame

In [7]:
def recognize_vehicle(video_path):
    cap = cv2.VideoCapture(video_path)
    
    target_time_ms = (3)*1000
    cap.set(cv2.CAP_PROP_POS_MSEC, target_time_ms)
    
    if not cap.isOpened():
        print("Cannot open camera")
        exit()
    
    while True:
        ret, raw_frame = cap.read()

        if not ret:
            print("cannot receive frame")
            break
    
        raw_frame = cv2.resize(raw_frame, (0,0), fx=0.35, fy=0.35, interpolation=cv2.INTER_AREA)
        
        # --- 改用新的追蹤函式 ---
        # 這裡會同時做 YOLO 追蹤 + 穩定點判斷
        result_frame = rear_vehicle_tracker(raw_frame)

        cv2.imshow("original frame", raw_frame)
        cv2.imshow("output", result_frame)
        
        if cv2.waitKey(1) == ESC:
            break

    cap.release()
    cv2.destroyAllWindows()

In [1]:
if __name__ == '__main__':
    recognize_vehicle(video_path)

NameError: name 'recognize_vehicle' is not defined